<a href="https://colab.research.google.com/github/Psynotch/firstrep/blob/master/Session7_Extended_Zero_Verification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
%pip -q install python-flint==0.9.0 mpmath==1.3.0

In [3]:
  from flint import acb, acb_series, arb, ctx
from fractions import Fraction
import mpmath as mp

ctx.prec = 192
mp.mp.dps = 60

print("SESSION 07 INITIALIZED")
print("FLINT precision:", ctx.prec, "bits")
print("mpmath precision:", mp.mp.dps, "decimal digits")

SESSION 07 INITIALIZED
FLINT precision: 192 bits
mpmath precision: 60 decimal digits


In [4]:
#Define full boundary
def fraction_ball(q):
    return arb(f"{q.numerator}/{q.denominator}")


def interval_ball(a, b):
    midpoint = (a + b) / 2
    radius = abs(b - a) / 2

    return arb(
        f"{midpoint.numerator}/{midpoint.denominator}",
        f"{radius.numerator}/{radius.denominator}"
    )


def segment_box(p, q):
    return acb(
        interval_ball(p[0], q[0]),
        interval_ball(p[1], q[1])
    )


# Counterclockwise contour:
# 0 <= real(s) <= 1, 10 <= imag(s) <= 100

corners = [
    (Fraction(0), Fraction(10)),
    (Fraction(1), Fraction(10)),
    (Fraction(1), Fraction(100)),
    (Fraction(0), Fraction(100)),
    (Fraction(0), Fraction(10))
]

MAX_DEPTH = 26

stats = {
    "certified_pieces": 0,
    "max_depth": 0,
    "unresolved": []
}


def certify_segment(p, q, depth=0):

    stats["max_depth"] = max(stats["max_depth"], depth)

    zeta_box = segment_box(p, q).zeta()
    lower = zeta_box.abs_lower()

    if lower.is_finite() and not lower.contains(0):
        stats["certified_pieces"] += 1
        return True

    if depth >= MAX_DEPTH:
        stats["unresolved"].append((p, q))
        return False

    midpoint = (
        (p[0] + q[0]) / 2,
        (p[1] + q[1]) / 2
    )

    left_ok = certify_segment(p, midpoint, depth + 1)
    right_ok = certify_segment(midpoint, q, depth + 1)

    return left_ok and right_ok


boundary_certified = True

print("SESSION 07: BOUNDARY CERTIFICATION")
print("----------------------------------")

for i in range(4):

    success = certify_segment(corners[i], corners[i + 1])

    print(
        f"Side {i + 1}:",
        "CERTIFIED" if success else "NOT CERTIFIED"
    )

    boundary_certified = boundary_certified and success


print("\nBoundary zero-free:", boundary_certified)
print("Certified subsegments:", stats["certified_pieces"])
print("Maximum depth:", stats["max_depth"])
print("Unresolved segments:", len(stats["unresolved"]))

if not boundary_certified:
    raise RuntimeError(
        "Boundary certification failed. Stop before integration."
    )

SESSION 07: BOUNDARY CERTIFICATION
----------------------------------
Side 1: CERTIFIED
Side 2: CERTIFIED
Side 3: CERTIFIED
Side 4: CERTIFIED

Boundary zero-free: True
Certified subsegments: 3571
Maximum depth: 13
Unresolved segments: 0


In [5]:
#Rigorous Contour integration
def zeta_log_derivative(z, analytic):
    jet = acb_series([z, 1], prec=2).zeta()
    coeff = jet.coeffs()

    return coeff[1] / coeff[0]


def interpolate_point(p, q, k, n):
    fraction = Fraction(k, n)

    return (
        p[0] + (q[0] - p[0]) * fraction,
        p[1] + (q[1] - p[1]) * fraction
    )


def to_acb(point):
    return acb(
        fraction_ball(point[0]),
        fraction_ball(point[1])
    )


def ceil_fraction(x):
    return (
        x.numerator + x.denominator - 1
    ) // x.denominator


MAX_PANEL_LENGTH = Fraction(1, 5)
tol = arb("1e-22")

side_integrals = []

print("SESSION 07: CONTOUR INTEGRATION")
print("--------------------------------")

for i in range(4):

    if not boundary_certified:
        raise RuntimeError("Boundary has not been certified.")

    p = corners[i]
    q = corners[i + 1]

    span = max(
        abs(q[0] - p[0]),
        abs(q[1] - p[1])
    )

    n = max(
        1,
        ceil_fraction(span / MAX_PANEL_LENGTH)
    )

    edge_total = acb(0)

    print(f"\nSide {i + 1}: {n} panels")

    for k in range(n):

        p1 = interpolate_point(p, q, k, n)
        p2 = interpolate_point(p, q, k + 1, n)

        result = acb.integral(
            zeta_log_derivative,
            to_acb(p1),
            to_acb(p2),
            abs_tol=tol,
            rel_tol=tol,
            deg_limit=80,
            eval_limit=200000,
            depth_limit=5000,
            use_heap=True
        )

        if not result.is_finite():
            raise RuntimeError(
                f"Integration failed: side {i + 1}, panel {k + 1}"
            )

        edge_total += result

    side_integrals.append(edge_total)

    print("Side integral:", edge_total)


total_integral = sum(side_integrals, acb(0))

print("\nTOTAL CONTOUR INTEGRAL")
print("----------------------")
print(total_integral)

SESSION 07: CONTOUR INTEGRATION
--------------------------------

Side 1: 5 panels
Side integral: [-0.232354013292327389218 +/- 1.30e-22] + [-0.021073270727252294451 +/- 4.92e-22]j

Side 2: 450 panels
Side integral: [0.15856807104401354242 +/- 9.54e-21] + [0.03710072003156357618 +/- 9.11e-21]j

Side 3: 5 panels
Side integral: [1.383646559789372942238 +/- 4.54e-22] + [0.069511263847732556323 +/- 4.00e-22]j

Side 4: 450 panels
Side integral: [-1.3098606175410590954 +/- 7.06e-20] + [182.1268351950559639928 +/- 4.98e-20]j

TOTAL CONTOUR INTEGRAL
----------------------
[+/- 3.55e-20] + [182.2123739082080078308 +/- 6.88e-20]j


In [6]:
#Determine exact zero count
two_pi_i = 2 * acb.pi() * acb(0, 1)

zero_count_ball = total_integral / two_pi_i

unique_count = zero_count_ball.unique_fmpz()

print("SESSION 07: ZERO COUNT")
print("----------------------")
print("Enclosure:", zero_count_ball)

if unique_count is None:

    print("\nNOT YET CERTIFIED")
    print("The enclosure does not isolate a unique integer.")

else:

    certified_full_count = int(unique_count)

    print("\nCERTIFIED ZERO COUNT:", certified_full_count)
    print("Expected count:", 29)
    print("Matches expected:", certified_full_count == 29)

SESSION 07: ZERO COUNT
----------------------
Enclosure: [29.00000000000000000000 +/- 5.65e-21] + [+/- 5.65e-21]j

CERTIFIED ZERO COUNT: 29
Expected count: 29
Matches expected: True


In [7]:
#Generate 29 candidate intervals
mp.mp.dps = 60

centers = []

n = 1

while True:

    gamma = mp.im(mp.zetazero(n))

    if gamma >= 100:
        break

    if gamma > 10:
        centers.append(
            Fraction(mp.nstr(gamma, 50))
        )

    n += 1


radius = Fraction(1, 100)

intervals = [
    (center - radius, center + radius)
    for center in centers
]

print("CANDIDATE INTERVALS")
print("-------------------")
print("Intervals generated:", len(intervals))

assert len(intervals) == 29, (
    f"Expected 29 intervals, generated {len(intervals)}"
)

assert all(
    intervals[i][1] < intervals[i + 1][0]
    for i in range(len(intervals) - 1)
)

assert all(
    left > 10 and right < 100
    for left, right in intervals
)

print("All intervals are disjoint: True")
print("All intervals inside (10,100): True")

CANDIDATE INTERVALS
-------------------
Intervals generated: 29
All intervals are disjoint: True
All intervals inside (10,100): True


In [9]:
#Rigorous Hardy Z-function verification
def hardy_z_enclosure(t_fraction):

    t = fraction_ball(t_fraction)

    s = acb(arb(1) / 2, t)

    gamma_arg = acb(
        arb(1) / 4,
        t / 2
    )

    theta = (
        gamma_arg.lgamma().imag
        - (t / 2) * arb.pi().log()
    )

    phase = acb(0, theta).exp()

    result = phase * s.zeta()

    return result.real, result.imag


def certified_sign(value):

    sign = value.sgn()

    if sign == arb(1):
        return "+"

    if sign == arb(-1):
        return "-"

    return "UNKNOWN"


verified = 0

print("SESSION 07: HARDY Z SIGN-CHANGE VERIFICATION")
print("---------------------------------------------")

for i, (left, right) in enumerate(intervals, 1):

    z_left, im_left = hardy_z_enclosure(left)
    z_right, im_right = hardy_z_enclosure(right)

    sign_left = certified_sign(z_left)
    sign_right = certified_sign(z_right)

    real_consistency = (
        im_left.contains(0)
        and im_right.contains(0)
    )

    sign_change = (
        sign_left != "UNKNOWN"
        and sign_right != "UNKNOWN"
        and sign_left != sign_right
    )

    passed = real_consistency and sign_change

    if passed:
        verified += 1

    print(
        f"Zero {i:02d}: "
        f"Z(left)={sign_left}, "
        f"Z(right)={sign_right}, "
        f"REAL={real_consistency}, "
        f"PASS={passed}"
    )


print("\nVERIFIED SIGN CHANGES:", verified)
print("TOTAL INTERVALS:", len(intervals))

SESSION 07: HARDY Z SIGN-CHANGE VERIFICATION
---------------------------------------------
Zero 01: Z(left)=-, Z(right)=+, REAL=True, PASS=True
Zero 02: Z(left)=+, Z(right)=-, REAL=True, PASS=True
Zero 03: Z(left)=-, Z(right)=+, REAL=True, PASS=True
Zero 04: Z(left)=+, Z(right)=-, REAL=True, PASS=True
Zero 05: Z(left)=-, Z(right)=+, REAL=True, PASS=True
Zero 06: Z(left)=+, Z(right)=-, REAL=True, PASS=True
Zero 07: Z(left)=-, Z(right)=+, REAL=True, PASS=True
Zero 08: Z(left)=+, Z(right)=-, REAL=True, PASS=True
Zero 09: Z(left)=-, Z(right)=+, REAL=True, PASS=True
Zero 10: Z(left)=+, Z(right)=-, REAL=True, PASS=True
Zero 11: Z(left)=-, Z(right)=+, REAL=True, PASS=True
Zero 12: Z(left)=+, Z(right)=-, REAL=True, PASS=True
Zero 13: Z(left)=-, Z(right)=+, REAL=True, PASS=True
Zero 14: Z(left)=+, Z(right)=-, REAL=True, PASS=True
Zero 15: Z(left)=-, Z(right)=+, REAL=True, PASS=True
Zero 16: Z(left)=+, Z(right)=-, REAL=True, PASS=True
Zero 17: Z(left)=-, Z(right)=+, REAL=True, PASS=True
Zero 18:

In [10]:
print("SESSION 07 FINAL VERIFICATION")
print("=============================")

print("Boundary certified:", boundary_certified)
print("Certified zero count:", unique_count)
print("Hardy sign changes:", verified)

success = (
    boundary_certified
    and unique_count is not None
    and int(unique_count) == 29
    and len(intervals) == 29
    and verified == 29
)

if success:

    print("\n================================")
    print("SESSION 07: SUCCESS")
    print("================================")

    print("Total zeros in rectangle: 29")
    print("Distinct critical-line zeros: 29")
    print("All zeros are simple.")
    print("All zeros lie on the critical line.")

else:

    print("\nSESSION 07: NOT YET COMPLETE")

    if unique_count is None:
        print("Zero count is not certified.")

    if verified != len(intervals):
        print("Not all intervals passed.")

SESSION 07 FINAL VERIFICATION
Boundary certified: True
Certified zero count: 29
Hardy sign changes: 29

SESSION 07: SUCCESS
Total zeros in rectangle: 29
Distinct critical-line zeros: 29
All zeros are simple.
All zeros lie on the critical line.
